# Retrieval evaluation 

Testing against our search options
- pgvector_search
- pg_full_text_search
- pg_full_text_search_soft_match
- rrf - reciprocal ranked fusion

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv('../data/processed/search_ground_truth.csv')

ground_truth = df_ground_truth.to_dict(orient="records")


In [35]:
from tools.retrieval import Retrieval, SearchHit, FusedHit
from embedder import Embedder
from utils.utils import get_connection
from tqdm.auto import tqdm

embedder = Embedder(path="../models/Xenova/all-MiniLM-L6-v2")
connection = get_connection()

In [36]:
def hit_id(result):
  return result.hit.id if isinstance(result, FusedHit) else result.id

def compute_relevance(q, search_fn, report_year=None):
  doc_id = q["document"]
  kwargs = {"query": q["question"]}
  if "report_year" in search_fn.__code__.co_varnames:
    kwargs["report_year"] = report_year
  results = search_fn(**kwargs)
  return [int(hit_id(r) == doc_id) for r in results]

def compute_relevance_total(ground_truth, search_fn, report_year=None):
  relevance_total = []
  for q in tqdm(ground_truth):
    relevance_total.append(compute_relevance(q, search_fn, report_year))
  return relevance_total


In [37]:
with get_connection() as connection:
    search = Retrieval(embedder, connection)
    relevance = compute_relevance_total(ground_truth, search.pgvector_search)
    print(f"Relevance for first query: {relevance}")

100%|██████████| 104/104 [00:00<00:00, 253.01it/s]

Relevance for first query: [[0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 1], [0, 1, 0, 0, 0], [0, 0, 0, 0, 1], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0], [0,

In [38]:
# hit rate if there is at least one relevant document per query
def hit_rate(relevance_total):
  return sum(1 for r in relevance_total if any(r)) / len(relevance_total)

hit_rate(relevance)


0.4423076923076923

In [39]:
def reciprocal_rank(line):
  for rank, relevance in enumerate(line):
    if relevance:
      return 1 / (rank + 1)
  return 0

def mrr(relevance_total):
  return sum(reciprocal_rank(line) for line in relevance_total) / len(relevance_total)

print(mrr(relevance))

0.2467948717948718


In [40]:
def evaluate(queries, search_method="pgvector_search", report_year=None):
  with get_connection() as connection:
    search = Retrieval(embedder, connection)
    search_fn = getattr(search, search_method)
    relevance_total = compute_relevance_total(queries, search_fn, report_year)

  return {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total),
  }



evaluate(ground_truth, "pgvector_search", 2025)

  0%|          | 0/104 [00:00<?, ?it/s]

100%|██████████| 104/104 [00:00<00:00, 291.36it/s]


{'hit_rate': 0.9423076923076923, 'mrr': 0.7193910256410256}

In [41]:
evaluate(ground_truth, "pg_full_text_search", 2025)

  0%|          | 0/104 [00:00<?, ?it/s]

100%|██████████| 104/104 [00:00<00:00, 1637.90it/s]


{'hit_rate': 0.22115384615384615, 'mrr': 0.21153846153846154}

In [42]:
evaluate(ground_truth, "pg_full_text_search_soft_match", 2025)

  0%|          | 0/104 [00:00<?, ?it/s]

100%|██████████| 104/104 [00:00<00:00, 823.35it/s]


{'hit_rate': 0.8846153846153846, 'mrr': 0.6956730769230769}

In [43]:
def compute_relevance_rrf(
  q,
  search,
  report_year=2025,
  fts_method="pg_full_text_search_soft_match",
  rrf_method="rrf",
  search_k=20,
  num_results=5,
):
  # Ground truth is 2025-only. Search both lists with the year filter,
  # then fuse a larger candidate pool down to num_results (same as main.py).
  doc_id = q["document"]
  search_kwargs = {
    "query": q["question"],
    "num_results": search_k,
    "report_year": report_year,
  }

  vector_hits = search.pgvector_search(**search_kwargs)
  fts_hits = getattr(search, fts_method)(**search_kwargs)
  if any(h.report_year != report_year for h in vector_hits + fts_hits):
    raise ValueError("search results included a year other than report_year")

  fused = getattr(search, rrf_method)(vector_hits, fts_hits, num_results=num_results)
  return [int(r.hit.id == doc_id) for r in fused]


def compute_relevance_total_rrf(
  queries,
  search,
  report_year=2025,
  fts_method="pg_full_text_search_soft_match",
  rrf_method="rrf",
  search_k=20,
  num_results=5,
):
  relevance_total = []
  for q in tqdm(queries):
    relevance_total.append(
      compute_relevance_rrf(
        q,
        search,
        report_year,
        fts_method=fts_method,
        rrf_method=rrf_method,
        search_k=search_k,
        num_results=num_results,
      )
    )
  return relevance_total


def evaluate_rrf(
  queries,
  report_year=2025,
  fts_method="pg_full_text_search_soft_match",
  rrf_method="rrf",
  search_k=20,
  num_results=5,
):
  with get_connection() as connection:
    search = Retrieval(embedder, connection)
    relevance_total = compute_relevance_total_rrf(
      queries,
      search,
      report_year,
      fts_method=fts_method,
      rrf_method=rrf_method,
      search_k=search_k,
      num_results=num_results,
    )

  return {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total),
  }


evaluate_rrf(ground_truth, 2025)

  0%|          | 0/104 [00:00<?, ?it/s]

100%|██████████| 104/104 [00:00<00:00, 203.73it/s]


{'hit_rate': 0.9519230769230769, 'mrr': 0.7681089743589744}